# Hull-White 1 facteur — calibration sur swaptions EUR ATM

Première brique d'un modèle gaussien à deux facteurs (taux + spread souverain). Pour l'instant $\sigma_y = 0$ : seul le facteur de taux $x(t)$ est modélisé.

$$r(t) = \alpha(t) + x(t), \qquad dx = -a\,x\,dt + \sigma\,dW, \qquad x(0) = 0$$


## 1. Données de marché

Deux feuilles dans le fichier Bloomberg :

- **`Worksheet`** : courbe €STR OIS — un taux swap par tenor (en %). Elle servira à construire $P^M(0,T)$ et $f^M(0,T)$.
- **`Sheet1`** : matrice de vols normales ATM (en bp), expiries en ligne, tenors du swap sous-jacent en colonne. Ce sont les instruments de calibration.

Les tenors Bloomberg (`1W`, `18M`, `10Y`, `1Mo`, `5Yr`) sont convertis en années pour tout le reste du notebook.


In [1]:
import numpy as np
import pandas as pd
from scipy.interpolate import CubicSpline
from scipy.optimize import minimize
from scipy.stats import norm

FICHIER = "../data/market data bloom.xlsx"

def tenor_en_annees(s):
    """'1W' -> 1/52, '18M' / '1Mo' -> 1.5 / 1/12, '10Y' / '10Yr' -> 10."""
    n = float("".join(c for c in s if c.isdigit()))
    return n / 52 if "W" in s else n / 12 if "M" in s else n

# --- courbe €STR : tenor (années) -> taux swap (décimal)
courbe = pd.read_excel(FICHIER, sheet_name="Worksheet", usecols=["Tenor", "Yield"])
courbe["T"] = courbe["Tenor"].map(tenor_en_annees)
courbe["taux"] = courbe["Yield"] / 100
courbe = courbe[["T", "taux"]]

# --- vols ATM normales : lignes = expiry (années), colonnes = tenor (années), valeurs en décimal
vols = pd.read_excel(FICHIER, sheet_name="Sheet1", index_col=0)
vols.index = [tenor_en_annees(s) for s in vols.index]
vols.columns = [tenor_en_annees(s) for s in vols.columns]
vols = vols / 1e4                      # bp -> décimal

print("Courbe €STR :", courbe.shape, "piliers de", courbe["T"].min(), "à", courbe["T"].max(), "ans")
print(courbe.head(8).T.to_string(), "\n")
print("Vols ATM :", vols.shape, "(expiries x tenors), en décimal")
print("expiries :", list(vols.index))
print("tenors   :", list(vols.columns))
print(vols.iloc[:5, :6].to_string())


Courbe €STR : (32, 2) piliers de 0.019230769230769232 à 50.0 ans
             0         1         2         3         4         5         6         7
T     0.019231  0.038462  0.083333  0.166667  0.250000  0.333333  0.416667  0.500000
taux  0.022260  0.023330  0.023951  0.024253  0.024536  0.024952  0.025330  0.025685 

Vols ATM : (21, 14) (expiries x tenors), en décimal
expiries : [0.08333333333333333, 0.16666666666666666, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 12.0, 15.0, 20.0, 25.0, 30.0]
tenors   : [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 15.0, 20.0, 25.0, 30.0]
              1.0      2.0      3.0      4.0      5.0      6.0
0.083333  0.00604  0.00729  0.00715  0.00684  0.00649  0.00630
0.166667  0.00640  0.00741  0.00725  0.00702  0.00678  0.00659
0.250000  0.00687  0.00764  0.00742  0.00720  0.00697  0.00679
0.500000  0.00752  0.00795  0.00768  0.00744  0.00722  0.00709
0.750000  0.00798  0.00820  0.00791  0.00767  0.00745  0.00732


## 2. Courbe €STR : bootstrap des zéro-coupons, $P^M(0,T)$ et $f^M(0,T)$

**Bootstrap.** Un swap OIS de maturité $T$ et de taux $S$ paie une jambe fixe annuelle aux dates $T, T-1, T-2, \dots$ (dernière date $>0$, donc une seule date $T$ si $T \le 1$). Sa valeur est nulle à l'initiation, ce qui donne l'équation de pricing

$$S \sum_{i} \tau_i\, P^M(0,T_i) + P^M(0,T_n) = 1, \qquad P^M(0,T) = e^{-R(T)\,T}.$$

On résout pilier par pilier : les $P^M$ aux dates de paiement antérieures sont déjà connus (interpolation linéaire du taux zéro $R$ entre piliers bootstrappés), la seule inconnue est $R(T)$, trouvée par `brentq`. Pour $T \le 1$ l'équation se réduit à $P = 1/(1+ST)$.

**Interpolation.** `CubicSpline` sur les taux zéro $R(T)$ : la courbe est $C^2$, donc le forward instantané est continu — indispensable, car $\alpha(t)$ (cellule 4) contient $f^M(0,t)$ et le taux court hériterait de tout saut.

$$P^M(0,T) = e^{-R(T)\,T}, \qquad f^M(0,T) = -\frac{\partial \log P^M(0,T)}{\partial T} = R(T) + T\,R'(T)$$

$R'$ est la dérivée analytique de la spline (`spline(T, 1)`), pas une différence finie.


In [2]:
from scipy.optimize import brentq

# --- bootstrap : taux swap OIS -> taux zéro continus, pilier par pilier
T_zc, R_zc = [], []
for T, S in zip(courbe["T"], courbe["taux"]):
    dates = np.arange(T, 0, -1)[::-1]                        # T, T-1, ..., > 0
    tau = np.diff(np.concatenate(([0.0], dates)))
    def residu(r):                                           # r = R(T), l'inconnue
        R = np.interp(dates, T_zc + [T], R_zc + [r])         # R aux dates de paiement
        P = np.exp(-R * dates)
        return S * np.sum(tau * P) + P[-1] - 1
    R_zc.append(brentq(residu, -0.1, 0.5))
    T_zc.append(T)
T_zc, R_zc = np.array(T_zc), np.array(R_zc)

# --- spline sur les taux zéro, P_mkt et f_mkt
spline = CubicSpline(T_zc, R_zc)

def P_mkt(T):
    return np.exp(-spline(T) * T)

def f_mkt(T):
    return spline(T) + T * spline(T, 1)

# --- vérifications
print("taux swap vs taux zéro bootstrappé (en %) :")
print(pd.DataFrame({"T": T_zc, "swap": courbe["taux"] * 100, "zéro": R_zc * 100})
        .set_index("T").loc[[1, 2, 5, 10, 20, 30, 50]].T.round(4).to_string())

T = np.linspace(0.1, 50, 500); h = 1e-5
f_fd = -(np.log(P_mkt(T + h)) - np.log(P_mkt(T - h))) / (2 * h)
print(f"\nmax |f_mkt - (-dlogP/dT) par diff. finies| = {np.max(np.abs(f_mkt(T) - f_fd)):.2e}")
print(f"f_mkt(0.5) = {f_mkt(0.5):.4%}   f_mkt(10) = {f_mkt(10):.4%}   f_mkt(30) = {f_mkt(30):.4%}")


taux swap vs taux zéro bootstrappé (en %) :
T       1.0     2.0     5.0     10.0   20.0    30.0    50.0
swap  2.7640  2.8816  2.9790  3.1574  3.361  3.3059  3.0440
zéro  2.7265  2.8425  2.9397  3.1285  3.356  3.2571  2.8112

max |f_mkt - (-dlogP/dT) par diff. finies| = 2.85e-11
f_mkt(0.5) = 2.7704%   f_mkt(10) = 3.5291%   f_mkt(30) = 2.7606%


## 3. Brique $H(a,t,T)$

$$H(a,t,T) = \frac{1 - e^{-a(T-t)}}{a} = \int_t^T e^{-a(u-t)}\,du$$

C'est l'intégrale du noyau de retour à la moyenne : un choc sur $x(t)$ se propage sur le taux court futur avec le poids $e^{-a(u-t)}$, donc l'intégrale $\int_t^T x(u)\,du$ — et par suite $\log P(t,T)$ — réagit au choc avec la sensibilité $H$. C'est la **duration stochastique** du zéro-coupon : $-\partial \log P(t,T) / \partial x = H(a,t,T)$.

Deux limites utiles : $H \to T - t$ quand $a \to 0$ (pas de retour à la moyenne, le choc est permanent) et $H \to 1/a$ quand $T - t \to \infty$ (le choc est absorbé, la sensibilité plafonne). C'est ce plafond qui empêche un modèle à un facteur de donner beaucoup de volatilité aux taux longs.

Implémentation avec `-np.expm1(-a·τ)/a` : `expm1` calcule $e^u - 1$ sans perte de précision quand $a\tau$ est petit.


In [3]:
def H(a, t, T):
    return -np.expm1(-a * (T - t)) / a

a_test = 0.05
maturites = np.array([1, 2, 5, 10, 20, 30, 50, 100])
print(f"a = {a_test}, plafond 1/a = {1 / a_test:.2f}")
print(pd.DataFrame({"T": maturites, "H(a,0,T)": H(a_test, 0, maturites), "T (limite a->0)": maturites})
        .set_index("T").T.round(3).to_string())
print(f"\nH(1e-8, 0, 10) = {H(1e-8, 0, 10):.6f}  (doit valoir 10 : limite a -> 0, expm1 garde la precision)")


a = 0.05, plafond 1/a = 20.00
T                  1      2      5       10      20      30      50       100
H(a,0,T)         0.975  1.903  4.424   7.869  12.642  15.537  18.358   19.865
T (limite a->0)  1.000  2.000  5.000  10.000  20.000  30.000  50.000  100.000

H(1e-8, 0, 10) = 10.000000  (doit valoir 10 : limite a -> 0, expm1 garde la precision)


## 4. Variance intégrée $V(t,T)$, recalage $\alpha(t)$ et $\int_0^T \alpha$

**$V(t,T)$** est la variance conditionnelle de l'intégrale du facteur, $V(t,T) = \mathrm{Var}\big[\int_t^T x(u)\,du \,\big|\, \mathcal F_t\big]$. Comme $x$ est gaussien, cette intégrale l'est aussi, et son espérance exponentielle ne dépend que de sa moyenne et de $V$ :

$$V(t,T) = \frac{\sigma^2}{a^2}\left[\tau + \frac{2}{a}e^{-a\tau} - \frac{1}{2a}e^{-2a\tau} - \frac{3}{2a}\right], \qquad \tau = T - t.$$

**$\int_0^T \alpha$.** Le modèle doit reproduire la courbe de marché : $P(0,T) = \mathbb E\big[e^{-\int_0^T r}\big] = e^{-\int_0^T \alpha}\,\mathbb E\big[e^{-\int_0^T x}\big] = e^{-\int_0^T \alpha + \frac12 V(0,T)}$. En imposant $P(0,T) = P^M(0,T)$ :

$$\int_0^T \alpha(u)\,du = -\log P^M(0,T) + \tfrac12 V(0,T).$$

**$\alpha(t)$** s'obtient en dérivant en $T$ : $-\partial_T \log P^M = f^M$ et $\tfrac12 \partial_T V(0,T) = \frac{\sigma^2}{2a^2}(1 - e^{-aT})^2$, d'où

$$\alpha(t) = f^M(0,t) + \frac{\sigma^2}{2a^2}\left(1 - e^{-at}\right)^2.$$

Le second terme est la correction de convexité : $\mathbb E[r(t)] = \alpha(t) > f^M(0,t)$, car l'actualisation est convexe en $r$ (inégalité de Jensen).

Les paramètres $a$ et $\sigma$ sont des **variables globales** : c'est ce que la calibration fera bouger.


In [4]:
a, sigma = 0.05, 0.01          # paramètres du modèle (globaux), point de départ de la calibration

def V(t, T):
    tau = T - t
    return sigma**2 / a**2 * (tau + 2 / a * np.exp(-a * tau) - np.exp(-2 * a * tau) / (2 * a) - 3 / (2 * a))

def alpha(t):
    return f_mkt(t) + sigma**2 / (2 * a**2) * (1 - np.exp(-a * t))**2

def int_alpha(T):
    return -np.log(P_mkt(T)) + 0.5 * V(0, T)

# --- vérification : alpha doit être la dérivée de int_alpha
t = np.array([0.5, 1, 2, 5, 10, 20, 30, 45]); h = 1e-5
alpha_fd = (int_alpha(t + h) - int_alpha(t - h)) / (2 * h)
print(pd.DataFrame({"t": t, "alpha(t)": alpha(t), "d/dT int_alpha": alpha_fd, "f_mkt": f_mkt(t),
                    "convexité": alpha(t) - f_mkt(t)}).set_index("t").T.to_string(float_format="{:.6f}".format))
print(f"\nmax |alpha - d/dT int_alpha| = {np.max(np.abs(alpha(t) - alpha_fd)):.2e}")
print(f"V(0, 10) = {V(0, 10):.6f}   ->   écart-type de int_0^10 x = {np.sqrt(V(0, 10)):.4f}")


t                  0.5      1.0      2.0      5.0      10.0     20.0     30.0     45.0
alpha(t)       0.027716 0.029996 0.029683 0.031978 0.038387 0.042074 0.039676 0.034102
d/dT int_alpha 0.027716 0.029996 0.029683 0.031978 0.038387 0.042074 0.039676 0.034102
f_mkt          0.027704 0.029948 0.029502 0.031000 0.035291 0.034082 0.027606 0.018096
convexité      0.000012 0.000048 0.000181 0.000979 0.003096 0.007992 0.012071 0.016006

max |alpha - d/dT int_alpha| = 1.87e-11
V(0, 10) = 0.023297   ->   écart-type de int_0^10 x = 0.1526


## 5. Prix zéro-coupon $P(t,T,x)$

$$P(t,T) = \mathbb E\Big[e^{-\int_t^T r(u)\,du}\,\Big|\,\mathcal F_t\Big] = e^{-\int_t^T \alpha}\;\mathbb E\Big[e^{-\int_t^T x(u)\,du}\,\Big|\,x(t) = x\Big]$$

Sachant $x(t) = x$, $\int_t^T x(u)\,du$ est gaussienne de moyenne $H(a,t,T)\,x$ (le choc $x$ décroît en $e^{-a(u-t)}$, cf. cellule 3) et de variance $V(t,T)$. L'espérance d'une exponentielle de gaussienne vaut $e^{-\text{moyenne} + \frac12\text{variance}}$, d'où

$$P(t,T,x) = \exp\!\Big(-\!\int_t^T \alpha(u)\,du \;-\; H(a,t,T)\,x \;+\; \tfrac12 V(t,T)\Big), \qquad \int_t^T \alpha = \int_0^T\alpha - \int_0^t \alpha.$$

Le prix est **log-affine en $x$** : c'est ce qui rendra les options sur obligations (presque) fermées. En $t = 0$, $x = 0$ et on retrouve $P(0,T) = e^{-\int_0^T\alpha + \frac12 V(0,T)} = P^M(0,T)$ par construction de $\alpha$.


In [5]:
def P(t, T, x):
    return np.exp(-(int_alpha(T) - int_alpha(t)) - H(a, t, T) * x + 0.5 * V(t, T))

# --- vérification 1 : recalage exact de la courbe de marché
T = np.array([0.5, 1, 2, 5, 10, 20, 30, 50])
print(pd.DataFrame({"T": T, "P(0,T,0)": P(0, T, 0), "P_mkt(T)": P_mkt(T)})
        .set_index("T").T.to_string(float_format="{:.8f}".format))
print(f"max |P(0,T,0) - P_mkt(T)| = {np.max(np.abs(P(0, T, 0) - P_mkt(T))):.2e}")

# --- vérification 2 : sensibilité au facteur, -dlogP/dx doit valoir H
h = 1e-6
print(f"\n-dlogP(2,10,x)/dx = {-(np.log(P(2, 10, h)) - np.log(P(2, 10, -h))) / (2 * h):.6f}"
      f"   H(a,2,10) = {H(a, 2, 10):.6f}")


T              0.5        1.0        2.0        5.0        10.0       20.0       30.0       50.0
P(0,T,0) 0.98732034 0.97310343 0.94473555 0.86330854 0.73135944 0.51109405 0.37639156 0.24521683
P_mkt(T) 0.98732034 0.97310343 0.94473555 0.86330854 0.73135944 0.51109405 0.37639156 0.24521683
max |P(0,T,0) - P_mkt(T)| = 5.55e-17

-dlogP(2,10,x)/dx = 6.593599   H(a,2,10) = 6.593599


## 6. Obligation à coupons et duration stochastique $D_x$

Une obligation à coupons est un portefeuille de zéro-coupons : flux $K_i$ aux dates $T_i$ (coupons, plus le nominal sur le dernier). Seuls les flux **postérieurs à $t$** comptent :

$$B(t,x) = \sum_{T_i > t} K_i\,P(t,T_i,x).$$

Sa sensibilité relative au facteur est la moyenne des durations $H$ des zéro-coupons, pondérée par leur poids dans le prix :

$$D_x(t,x) = -\frac{1}{B}\frac{\partial B}{\partial x} = \frac{\sum_{T_i > t} K_i\,P(t,T_i,x)\,H(a,t,T_i)}{\sum_{T_i > t} K_i\,P(t,T_i,x)}.$$

La formule découle directement de $\partial_x P(t,T_i,x) = -H(a,t,T_i)\,P(t,T_i,x)$ (cellule 5). Contrairement au zéro-coupon, $D_x$ **dépend de $x$** : quand les taux montent, les flux lointains pèsent moins et la duration raccourcit. C'est cette dépendance qui empêche une formule exactement fermée pour l'option sur obligation à coupons, et qu'on gèlera à la cellule 7.


In [6]:
def B(t, T_i, K_i, x):
    apres = T_i > t
    return np.sum(K_i[apres] * P(t, T_i[apres], x))

def D_x(t, T_i, K_i, x):
    apres = T_i > t
    poids = K_i[apres] * P(t, T_i[apres], x)
    return np.sum(poids * H(a, t, T_i[apres])) / np.sum(poids)

# --- obligation test : 10 ans, coupon annuel 3 %, nominal 1
T_i = np.arange(1.0, 11.0)
K_i = np.full(10, 0.03); K_i[-1] += 1

# --- vérification : D_x doit valoir -(1/B) dB/dx par différences finies
h = 1e-6
for t, x in [(0, 0.0), (2.5, 0.0), (2.5, 0.02), (7, -0.01)]:
    fd = -(B(t, T_i, K_i, x + h) - B(t, T_i, K_i, x - h)) / (2 * h) / B(t, T_i, K_i, x)
    print(f"t = {t:<4} x = {x:>5}   B = {B(t, T_i, K_i, x):.6f}   D_x = {D_x(t, T_i, K_i, x):.6f}"
          f"   -(1/B) dB/dx = {fd:.6f}   H(a,t,10) = {H(a, t, 10):.4f}")


t = 0    x =   0.0   B = 0.986612   D_x = 6.985477   -(1/B) dB/dx = 6.985477   H(a,t,10) = 7.8694
t = 2.5  x =   0.0   B = 0.992788   D_x = 5.639842   -(1/B) dB/dx = 5.639842   H(a,t,10) = 6.2542
t = 2.5  x =  0.02   B = 0.887280   D_x = 5.595295   -(1/B) dB/dx = 5.595295   H(a,t,10) = 6.2542
t = 7    x = -0.01   B = 1.007117   D_x = 2.708315   -(1/B) dB/dx = 2.708315   H(a,t,10) = 2.7858


## 7. Volatilité intégrée $\Sigma_B(t,T_0,T_n)$

L'option (cellule 8) porte sur le **prix forward** de l'obligation, $F(u) = B(u)/P(u,T_0)$, restreint aux flux postérieurs à $T_0$. Sous la mesure $T_0$-forward c'est une martingale, et sa volatilité instantanée est $\sigma$ fois sa duration **forward** : la duration de $B$ moins celle du numéraire $P(u,T_0)$,

$$\sigma_F(u) = \sigma\,\big[D_x(u) - H(a,u,T_0)\big], \qquad D_x(u) = \sum_{T_i > T_0} w_i\,H(a,u,T_i).$$

La variance totale de $\log F(T_0)$ est l'intégrale de $\sigma_F^2$ sur la vie de l'option :

$$\Sigma_B^2(t,T_0,T_n) = \int_t^{T_0} \sigma^2\,\big[D_x(u) - H(a,u,T_0)\big]^2\,du.$$

**Gel des poids.** Les $w_i = K_i P(t,T_i,x) / \sum_j K_j P(t,T_j,x)$ dépendent en toute rigueur de $x(u)$ ; on les fige à leur valeur en $t$. C'est l'approximation standard qui rend $F(T_0)$ lognormal et permet Black. Intégrale par Gauss-Legendre : les nœuds $u_k \in [-1,1]$ sont ramenés sur $[t,T_0]$ et les poids multipliés par $(T_0-t)/2$.

Remarque : avec les poids gelés, $H(a,u,T_i) - H(a,u,T_0) = e^{-a(T_0-u)}H(a,T_0,T_i)$ et l'intégrale est en fait fermée :
$\Sigma_B^2 = \sigma^2 \bar H^2 \,\frac{1 - e^{-2a(T_0-t)}}{2a}$ avec $\bar H = \sum w_i H(a,T_0,T_i)$. Ça sert de vérification ; la quadrature restera utile avec le second facteur.


In [7]:
def Sigma_B(t, T0, T_i, K_i, x, n=32):
    apres = T_i > T0
    poids = K_i[apres] * P(t, T_i[apres], x)
    poids = poids / np.sum(poids)                            # gelés en t
    u, w = np.polynomial.legendre.leggauss(n)
    u, w = t + (T0 - t) * (u + 1) / 2, w * (T0 - t) / 2     # [-1, 1] -> [t, T0]
    D_fwd = np.sum(poids * (H(a, u[:, None], T_i[apres]) - H(a, u[:, None], T0)), axis=1)
    return np.sqrt(np.sum(w * sigma**2 * D_fwd**2))

# --- vérification 1 : forme fermée (poids gelés)
t, T0 = 0.0, 5.0
apres = T_i > T0
poids = K_i[apres] * P(t, T_i[apres], 0); poids /= poids.sum()
H_bar = np.sum(poids * H(a, T0, T_i[apres]))
ferme = sigma * H_bar * np.sqrt(-np.expm1(-2 * a * (T0 - t)) / (2 * a))
print(f"option 5 ans sur l'obligation 10 ans :  Sigma_B (Gauss-Legendre) = {Sigma_B(t, T0, T_i, K_i, 0):.8f}"
      f"   forme fermée = {ferme:.8f}")

# --- vérification 2 : un seul flux -> vol de l'option sur zéro-coupon de Hull-White
zc = sigma * H(a, T0, 10) * np.sqrt(-np.expm1(-2 * a * (T0 - t)) / (2 * a))
print(f"zéro-coupon 10 ans, option 5 ans :      Sigma_B = {Sigma_B(t, T0, np.array([10.0]), np.array([1.0]), 0):.8f}"
      f"   sigma_P Hull-White = {zc:.8f}")


option 5 ans sur l'obligation 10 ans :  Sigma_B (Gauss-Legendre) = 0.08297576   forme fermée = 0.08297576
zéro-coupon 10 ans, option 5 ans :      Sigma_B = 0.08775443   sigma_P Hull-White = 0.08775443


## 8. Option sur obligation à coupons : formule de Black

Option d'expiry $T_0$ et de strike $X$ sur l'obligation. À $T_0$ le sous-jacent vaut $B(T_0) = \sum_{T_i > T_0} K_i P(T_0,T_i)$, le payoff du call est $(B(T_0) - X)^+$. Sous la mesure $T_0$-forward (numéraire $P(\cdot,T_0)$) :

$$\text{Call}(t) = P(t,T_0)\;\mathbb E^{T_0}\big[(B(T_0) - X)^+\big].$$

Avec les poids gelés, $B(T_0)/P(T_0,T_0) = B(T_0)$ est lognormal de variance $\Sigma_B^2$ et d'espérance le forward $F/P(t,T_0)$ où $F = \sum_{T_i>T_0} K_i P(t,T_i,x)$ est la valeur aujourd'hui des flux livrés. L'espérance se calcule comme dans Black-Scholes, et en multipliant par $P(t,T_0)$ tout s'écrit en valeurs actualisées :

$$\text{Call} = F\,\Phi(d_1) - K\,\Phi(d_2), \qquad K = X\,P(t,T_0), \qquad d_{1,2} = \frac{\log(F/K) \pm \tfrac12\Sigma_B^2}{\Sigma_B}.$$

Le put s'obtient par parité, $\text{Put} = \text{Call} - F + K$ (détenir un call et vendre un put revient à acheter le forward).


In [8]:
def option_oblig(t, T0, T_i, K_i, X, x, put=False):
    F = np.sum(K_i[T_i > T0] * P(t, T_i[T_i > T0], x))
    K = X * P(t, T0, x)
    S = Sigma_B(t, T0, T_i, K_i, x)
    d1 = (np.log(F / K) + 0.5 * S**2) / S
    call = F * norm.cdf(d1) - K * norm.cdf(d1 - S)
    return call - F + K if put else call

# --- vérification : option 5 ans sur l'obligation 10 ans, strike 1
call, put = option_oblig(0, 5, T_i, K_i, 1.0, 0), option_oblig(0, 5, T_i, K_i, 1.0, 0, put=True)
F, K = np.sum(K_i[T_i > 5] * P(0, T_i[T_i > 5], 0)), 1.0 * P(0, 5, 0)
print(f"F = {F:.6f}   K = X P(0,5) = {K:.6f}   forward B(5) = F/P(0,5) = {F / K:.6f}")
print(f"call = {call:.6f}   put = {put:.6f}   call - put = {call - put:.6f}   F - K = {F - K:.6f}")

# limite sigma -> 0 : le call vaut la valeur intrinsèque actualisée (F - K)^+, strike 0.95 pour être dans la monnaie
sigma_sauve, sigma = sigma, 1e-6
print(f"sigma -> 0, X = 0.95 :   call = {option_oblig(0, 5, T_i, K_i, 0.95, 0):.6f}"
      f"   (F - X P(0,5))^+ = {max(F - 0.95 * P(0, 5, 0), 0):.6f}")
sigma = sigma_sauve


F = 0.848957   K = X P(0,5) = 0.863309   forward B(5) = F/P(0,5) = 0.983376
call = 0.021732   put = 0.036084   call - put = -0.014352   F - K = -0.014352
sigma -> 0, X = 0.95 :   call = 0.028814   (F - X P(0,5))^+ = 0.028814


## 9. Swaption = option sur la jambe fixe

Une swaption payeuse d'expiry $T_0$ sur un swap de tenor $n$ (jambe fixe annuelle au taux $S$, dates $T_i = T_0 + i$) donne le droit de payer $S$ et recevoir le flottant. À $T_0$, la jambe flottante vaut le pair (1) et la jambe fixe vaut l'obligation $B(T_0) = \sum_i S\,\tau_i\,P(T_0,T_i) + P(T_0,T_n)$. Le payoff est donc

$$\big(1 - B(T_0)\big)^+ \quad\Longrightarrow\quad \text{payeuse} = \text{put sur } B \text{ de strike } X = 1,\qquad \text{receveuse} = \text{call}.$$

D'où l'échéancier : $K_i = S\,\tau_i$ pour $i < n$, $K_n = S\,\tau_n + 1$. Le taux ATM est le taux forward du swap, celui qui annule sa valeur aujourd'hui :

$$S_{\text{ATM}} = \frac{P(0,T_0) - P(0,T_n)}{A}, \qquad A = \sum_i \tau_i\,P(0,T_i) \;\;(\text{annuité}).$$

À l'ATM, $B$ forward vaut exactement 1, donc $F = K$ et payeuse = receveuse.


In [9]:
def echeancier(T0, tenor, S):
    T_i = T0 + np.arange(1.0, tenor + 1)                     # jambe fixe annuelle
    K_i = S * np.diff(np.concatenate(([T0], T_i)))           # S * tau_i
    K_i[-1] += 1
    return T_i, K_i

def annuite(T0, tenor):
    T_i = T0 + np.arange(1.0, tenor + 1)
    return np.sum(np.diff(np.concatenate(([T0], T_i))) * P_mkt(T_i))

def taux_atm(T0, tenor):
    return (P_mkt(T0) - P_mkt(T0 + tenor)) / annuite(T0, tenor)

def swaption(T0, tenor, S, payeuse=True):
    T_i, K_i = echeancier(T0, tenor, S)
    return option_oblig(0, T0, T_i, K_i, 1.0, 0, put=payeuse)

# --- vérification : à l'ATM le forward de l'obligation vaut 1 et payeuse = receveuse
for T0, tenor in [(1, 10), (5, 5), (10, 10)]:
    S = taux_atm(T0, tenor)
    T_i, K_i = echeancier(T0, tenor, S)
    fwd = np.sum(K_i * P(0, T_i, 0)) / P(0, T0, 0)
    print(f"{T0:>2}Y x {tenor:>2}Y   S_atm = {S:.4%}   B forward = {fwd:.10f}"
          f"   payeuse = {swaption(T0, tenor, S):.6f}   receveuse = {swaption(T0, tenor, S, payeuse=False):.6f}")


 1Y x 10Y   S_atm = 3.2398%   B forward = 1.0000000000   payeuse = 0.026242   receveuse = 0.026242
 5Y x  5Y   S_atm = 3.3661%   B forward = 1.0000000000   payeuse = 0.028399   receveuse = 0.028399
10Y x 10Y   S_atm = 3.6509%   B forward = 1.0000000000   payeuse = 0.050053   receveuse = 0.050053


## 10. Prix de marché (Bachelier) et premier ordre de grandeur du modèle

Les vols Bloomberg sont des **vols normales** (Bachelier) : le taux swap forward est supposé gaussien, $dS = \sigma_N\,dW$. Le prix d'une swaption ATM (strike = forward) est alors, en unités de nominal,

$$\text{Prix}_{\text{ATM}} = A\;\sigma_N\,\sqrt{\frac{T_0}{2\pi}}, \qquad A = \sum_i \tau_i P^M(0,T_i),$$

car $\mathbb E[(S_{T_0} - S_0)^+] = \sigma_N\sqrt{T_0}\,\mathbb E[Z^+] = \sigma_N\sqrt{T_0}/\sqrt{2\pi}$, et l'annuité $A$ actualise et somme les flux du swap sur lesquels porte la différence de taux.

On construit une table longue (une ligne par swaption) en **excluant** les swaptions dont $T_0 + \text{tenor} > 50$ ans : au-delà du dernier pilier le spline extrapole sans contrôle (cellule 2). Le modèle est évalué au point de départ $(a, \sigma) = (0.05, 0.01)$ pour vérifier l'ordre de grandeur avant d'optimiser.


In [10]:
def prix_bachelier(T0, tenor, vol):
    return annuite(T0, tenor) * vol * np.sqrt(T0 / (2 * np.pi))

swaptions = vols.stack().reset_index()
swaptions.columns = ["T0", "tenor", "vol"]
swaptions = swaptions[swaptions["T0"] + swaptions["tenor"] <= 50].reset_index(drop=True)
swaptions["prix_mkt"] = [prix_bachelier(T0, n, v) for T0, n, v in zip(swaptions["T0"], swaptions["tenor"], swaptions["vol"])]

def prix_modele(swaptions):
    return np.array([swaption(T0, n, taux_atm(T0, n)) for T0, n in zip(swaptions["T0"], swaptions["tenor"])])

a, sigma = 0.05, 0.01
swaptions["prix_mod"] = prix_modele(swaptions)
swaptions["ratio"] = swaptions["prix_mod"] / swaptions["prix_mkt"]

print(f"{len(swaptions)} swaptions retenues sur {vols.size} (exclues : T0 + tenor > 50 ans)")
print(f"\nratio prix modèle / prix marché à (a, sigma) = ({a}, {sigma}) — lignes = expiry, colonnes = tenor :")
print(swaptions.pivot(index="T0", columns="tenor", values="ratio").round(2).to_string())
print(f"\nratio moyen = {swaptions['ratio'].mean():.3f}   min = {swaptions['ratio'].min():.3f}   max = {swaptions['ratio'].max():.3f}")
swaptions


291 swaptions retenues sur 294 (exclues : T0 + tenor > 50 ans)

ratio prix modèle / prix marché à (a, sigma) = (0.05, 0.01) — lignes = expiry, colonnes = tenor :
tenor      1.0   2.0   3.0   4.0   5.0   6.0   7.0   8.0   9.0   10.0  15.0  20.0  25.0  30.0
T0                                                                                           
0.083333   1.66  1.34  1.33  1.36  1.40  1.41  1.42  1.42  1.41  1.41  1.35  1.29  1.24  1.18
0.166667   1.56  1.32  1.31  1.33  1.34  1.35  1.35  1.35  1.35  1.34  1.29  1.24  1.19  1.14
0.250000   1.45  1.27  1.28  1.29  1.30  1.31  1.30  1.30  1.29  1.29  1.23  1.18  1.13  1.10
0.500000   1.32  1.22  1.23  1.24  1.25  1.24  1.24  1.23  1.22  1.21  1.15  1.10  1.05  1.00
0.750000   1.24  1.17  1.19  1.20  1.20  1.20  1.19  1.19  1.18  1.17  1.10  1.05  1.00  0.95
1.000000   1.19  1.15  1.16  1.17  1.17  1.16  1.16  1.15  1.14  1.14  1.07  1.01  0.96  0.91
1.500000   1.16  1.15  1.15  1.15  1.14  1.14  1.13  1.12  1.11  1.10  1.03  0.98  0.9

,T0,tenor,vol,prix_mkt,prix_mod,ratio
0,0.083333,1.0,0.00604,0.000675,0.001119,1.656932
1,0.083333,2.0,0.00729,0.001606,0.002153,1.340405
2,0.083333,3.0,0.00715,0.002329,0.003108,1.334595
3,0.083333,4.0,0.00684,0.002927,0.003989,1.362838
4,0.083333,5.0,0.00649,0.003421,0.004801,1.403539
...,...,...,...,...,...,...
286,30.000000,8.0,0.00596,0.034926,0.028113,0.804914
287,30.000000,9.0,0.00589,0.038379,0.030602,0.797357
288,30.000000,10.0,0.00586,0.041943,0.032924,0.784963
289,30.000000,15.0,0.00557,0.056677,0.042483,0.749555


## 11. Calibration de $(a, \sigma)$

On cherche les paramètres qui rapprochent au mieux les prix modèle des prix de marché, en **erreur relative** pour que les swaptions bon marché (expiries courtes) pèsent autant que les chères :

$$(a^*, \sigma^*) = \arg\min_{a,\sigma}\; \sqrt{\sum_k \left(\frac{\text{Prix}^{\text{mod}}_k(a,\sigma) - \text{Prix}^{\text{mkt}}_k}{\text{Prix}^{\text{mkt}}_k}\right)^2}.$$

L'objectif écrit dans les globales `a, sigma` puis repricie les 291 swaptions : chaque évaluation recalcule $\alpha$, $V$, $P$ et $\Sigma_B$ avec les nouveaux paramètres — rien n'est mis en cache, c'est volontaire pour la lisibilité. `L-BFGS-B` gère les bornes ; le gradient est estimé par différences finies par `scipy`.

Bornes : $a \in [0.001, 1]$, $\sigma \in [10^{-4}, 0.05]$ ; départ $(0.05, 0.01)$.


In [11]:
def objectif(params):
    global a, sigma
    a, sigma = params
    return np.sqrt(np.sum(((prix_modele(swaptions) - swaptions["prix_mkt"]) / swaptions["prix_mkt"])**2))

depart = [0.05, 0.01]
objectif_depart = objectif(depart)
resultat = minimize(objectif, x0=depart, method="L-BFGS-B", bounds=[(0.001, 1), (0.0001, 0.05)])
a, sigma = resultat.x                   # les globales gardent le point optimal

print(resultat.message, "-", resultat.nfev, "évaluations")
print(f"a* = {a:.5f}   sigma* = {sigma:.5%}   objectif = {resultat.fun:.4f}   (départ : {objectif_depart:.4f})")
print(f"RMSE relatif = {resultat.fun / np.sqrt(len(swaptions)):.2%}")


CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH - 57 évaluations
a* = 0.01380   sigma* = 0.73609%   objectif = 1.2281   (départ : 3.0182)
RMSE relatif = 7.20%


## 12. Diagnostics

Trois questions après une calibration :

1. **Qualité du fit** : RMSE relatif $\sqrt{\tfrac1N\sum_k \varepsilon_k^2}$ avec $\varepsilon_k = \text{Prix}^{\text{mod}}_k / \text{Prix}^{\text{mkt}}_k - 1$, et la **carte des résidus** expiry × tenor. Avec deux paramètres pour 291 instruments, le fit ne peut pas être uniforme : la carte montre *où* le modèle à un facteur pèche (structurellement : expiries courtes vs longues, tenors courts vs longs).
2. **Vol implicite modèle** : on inverse la formule de Bachelier ATM, $\sigma_N^{\text{mod}} = \text{Prix}^{\text{mod}} / (A\sqrt{T_0/2\pi})$, pour lire les résidus en bp de vol — l'unité des traders.
3. **Stabilité** : l'objectif n'a aucune raison d'être convexe. On relance `L-BFGS-B` depuis 5 points de départ dispersés dans les bornes ; si tous convergent vers le même $(a^*, \sigma^*)$, l'optimum est global (au moins dans la zone raisonnable).


In [12]:
# --- 1. résidus par swaption au point calibré
swaptions["prix_mod"] = prix_modele(swaptions)
swaptions["residu"] = swaptions["prix_mod"] / swaptions["prix_mkt"] - 1
swaptions["vol_mod"] = swaptions["vol"] * swaptions["prix_mod"] / swaptions["prix_mkt"]     # Bachelier ATM : prix linéaire en vol
swaptions["residu_bp"] = (swaptions["vol_mod"] - swaptions["vol"]) * 1e4

print(f"a* = {a:.5f}   sigma* = {sigma:.4%}   RMSE relatif = {np.sqrt(np.mean(swaptions['residu']**2)):.2%}"
      f"   |résidu| max = {swaptions['residu'].abs().max():.1%}   RMSE vol = {np.sqrt(np.mean(swaptions['residu_bp']**2)):.1f} bp")
print("\nrésidus prix (modèle / marché - 1, en %) — lignes = expiry, colonnes = tenor :")
print((swaptions.pivot(index="T0", columns="tenor", values="residu") * 100).round(1).to_string())
print("\nrésidus vol (vol modèle - vol marché, en bp) :")
print(swaptions.pivot(index="T0", columns="tenor", values="residu_bp").round(1).to_string())

# --- 2. stabilité : 5 points de départ
print("\nstabilité de l'optimum selon le point de départ :")
for depart in [[0.05, 0.01], [0.001, 0.0001], [0.5, 0.03], [0.2, 0.002], [0.01, 0.05]]:
    r = minimize(objectif, x0=depart, method="L-BFGS-B", bounds=[(0.001, 1), (0.0001, 0.05)])
    print(f"  départ (a, sigma) = ({depart[0]:<6}, {depart[1]:<6})  ->  a* = {r.x[0]:.5f}   sigma* = {r.x[1]:.5%}"
          f"   objectif = {r.fun:.5f}   {r.nfev} éval.")
a, sigma = resultat.x                   # on remet l'optimum de la cellule 11 dans les globales


a* = 0.01380   sigma* = 0.7361%   RMSE relatif = 7.20%   |résidu| max = 27.7%   RMSE vol = 4.5 bp

résidus prix (modèle / marché - 1, en %) — lignes = expiry, colonnes = tenor :
tenor      1.0   2.0   3.0   4.0   5.0   6.0   7.0   8.0   9.0   10.0  15.0  20.0  25.0  30.0
T0                                                                                           
0.083333   24.4   2.4   3.7   7.7  12.8  15.4  17.4  19.3  20.7  21.9  25.2  26.9  27.7  27.4
0.166667   17.4   0.7   2.3   4.9   7.9  10.3  12.1  13.8  15.4  16.4  19.3  21.5  22.6  23.0
0.250000    9.3  -2.4  -0.1   2.3   4.9   7.0   8.3   9.6  10.7  11.8  13.9  16.4  17.3  18.2
0.500000   -0.2  -6.3  -3.6  -1.2   1.2   2.3   3.4   4.5   5.3   6.1   7.0   8.7   9.2   8.9
0.750000   -6.1  -9.3  -6.6  -4.3  -2.1  -1.1  -0.1   1.0   1.9   3.0   3.3   4.6   4.7   3.8
1.000000   -9.0 -10.6  -8.0  -6.4  -4.6  -3.6  -2.6  -1.5  -0.7   0.2   0.4   0.8   0.7  -0.4
1.500000  -10.3 -10.3  -8.2  -7.0  -5.8  -4.8  -4.0  -3.3  -2.7  -2.1 